<a href="https://colab.research.google.com/github/RamziRBM/lab-sql-query-from-table-names/blob/main/lab-sql-query-from-table-names.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
from google.colab import userdata
import os

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [4]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': ['employees', 'salary', 'studies'],
        'definition': ['Employee information, name...',
                       'Salary details for each year',
                       'Educational studies, name of the institution, type of studies, level']}
df = pd.DataFrame(data)
print(df)

       table                                         definition
0  employees                      Employee information, name...
1     salary                       Salary details for each year
2    studies  Educational studies, name of the institution, ...


In [5]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [6]:
print(text_tables)

employees: Employee information, name...
salary: Salary details for each year
studies: Educational studies, name of the institution, type of studies, level


In [7]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [9]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="Which employees have studied at a university")

In [ ]:
print(return_OAI(pqt1))

```json
{
    "tables": ["employees", "salary"]
}
```


In [10]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="which employees are paid")

In [11]:
print(return_OAI(pqt3))

```json
{
    "tables": ["employees", "salary"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [12]:
pqt4 = prompt_question_tables.format(tables=text_tables,
                                     question="count the number of employees in each department")
print(return_OAI(pqt4))

```json
{
    "tables": ["employees"]
}
```


In [13]:
pqt5 = prompt_question_tables.format(tables=text_tables,
                                     question="Return the Education Institution with the highest level")
print(return_OAI(pqt5))

```json
{
    "tables": ["studies"]
}
```


In [14]:
authors_data = {'table': ['authors', 'titles', 'publisher', 'sales'],
        'definition': ['list of book authors with names and other personal data', 'titles of each author and year of publication', 'publisher of each book', 'sales of each book per month, quantities and sales income']}
dfa = pd.DataFrame(authors_data)
print(dfa)

       table                                         definition
0    authors  list of book authors with names and other pers...
1     titles      titles of each author and year of publication
2  publisher                             publisher of each book
3      sales  sales of each book per month, quantities and s...


In [15]:
text_tables_authors = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in dfa.iterrows()])
print(text_tables_authors)

prompt_question_tables_authors = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""

pqt = prompt_question_tables_authors.format(tables=text_tables_authors, question='what is the top selling author and how many books did they sell?')
print(return_OAI(pqt))

authors: list of book authors with names and other personal data
titles: titles of each author and year of publication
publisher: publisher of each book
sales: sales of each book per month, quantities and sales income
{
    "tables": ["authors", "titles", "sales"]
}
